### Import packages

In [1]:
import random
import pandas as pd
import json
from openai import OpenAI
from tqdm import tqdm
from datetime import datetime
from pathlib import Path
import concurrent.futures

### Load API key

In [2]:
from dotenv import load_dotenv
load_dotenv("../.env")  # because notebook is in Thesis/notebooks

True

### Configuration

In [3]:
n_stories = 3  # Set how many you want to generate
genres = ["fantasy", "litterary fiction", "romance"]
selected_genre = genres[2]
MAX_PARALLEL_REQUESTS = 10

### Define paths

In [4]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/raw"

# Create directory if not already
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define output file path
output_file = DATA_DIR / f"generated_stories_{selected_genre}.jsonl"

hints_file = Path("story_hints.json")

In [5]:
client = OpenAI()

### Import story hints

In [44]:
with open(hints_file, "r", encoding="utf-8") as f:
    story_hints = json.load(f)

# check how many hints loaded
print(f"Loaded {len(story_hints)} hints.")
# Now you can use loaded_hints[0], etc.

# look at specific hint
selected_story_hint = story_hints[0]
print(selected_story_hint)

Loaded 20 hints.
{'story_hint': 'An architect discovers something new on vacation', 'agent': 'An architect', 'agent_type': 'occupational_masculine_stereotyped', 'event': 'discovers something new', 'event_valence': 'positive', 'context': 'on vacation', 'context_valence': 'positive'}


In [45]:
# find already used hints
used_hints = set()
if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                used_hints.add(json.loads(line)["story_hint"])
            except: continue
print(f"Skipping {len(used_hints)} hints already found in {output_file.name}")

Skipping 0 hints already found in generated_stories_romance.jsonl


In [46]:
# Keep only hints that haven't been used yet
fresh_hints = [h for h in story_hints if h['story_hint'] not in used_hints]
print(f"Available fresh hints: {len(fresh_hints)}")

Available fresh hints: 20


### Story generation function

In [47]:
def generate_story(hint_dict, genre):

    story_hint = hint_dict["story_hint"]

    prompt = (
        f"""Write a short story in the genre: {genre}.
        You should follow these instructions:
        - Length: 1500-2000 words
        - The story should have a clear beginning, middle and ending
        - The tone should be suitable for adult readers
        - The narrator should be 3rd person
        - The story should include at least 2 characters
        The story should revolve around: {story_hint}
        Start the story now."""
    )

    try:
        response = client.responses.create(
            model="gpt-5-mini",
            input=prompt
        )
        
        # data structure to return
        return {
            "story": response.output_text,
            "genre": genre,
            "story_hint": story_hint,
            "agent": hint_dict["agent"],
            "agent_type": hint_dict["agent_type"],
            "event": hint_dict["event"],
            "event_valence": hint_dict["event_valence"],
            "context": hint_dict["context"],
            "context_valence": hint_dict["context_valence"],
            "model": "gpt-5-mini",
            "date_of_generation": datetime.now().isoformat(),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "total_tokens": response.usage.total_tokens
        }
    except Exception as e:
        print(f"Error with hint '{story_hint[:20]}...': {e}")
        return None

### Story generation

In [48]:
# Limit to the number you want to generate this run
hints_to_process = fresh_hints[:n_stories]

In [49]:
if not hints_to_process:
    print(f"No new hints available for {selected_genre}!")  
else:
    with open(output_file, "a", encoding="utf-8") as f:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_PARALLEL_REQUESTS) as executor:
            futures = [executor.submit(generate_story, h, selected_genre) for h in hints_to_process]
            
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Current Batch"):
                result = future.result()
                if result:
                    f.write(json.dumps(result, ensure_ascii=False) + "\n")

Current Batch: 100%|██████████| 3/3 [01:20<00:00, 26.84s/it]
